# 01 — Парсер Auto.ru

Этот ноутбук запускает скрапер каталога и карточек объявлений на Auto.ru.

**Важно: текущий датасет (`data/raw/autoru_raw.parquet`) трогать нельзя.**
По умолчанию ноутбук:
1. **Не запускает** скрапинг (`RUN_SCRAPE = False`).
2. Если его включить — пишет результаты в `data/raw/notebook_run/`,
   не затрагивая старый `autoru_raw.*`.

Шаги:
1. Импорты и проверка окружения.
2. Настройка `ScrapeConfig` с безопасными путями.
3. (Опционально) запуск скрапинга.
4. Просмотр результата.


In [29]:
# Allow notebooks to import the project's `functions` package.
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/demn/autoru_market_research


## 1. Импорты и логирование

`functions.scraper.run_scrape` оборачивает `AutoRuScraper` контекст-менеджером,
который сохраняет состояние резюмирования (state-файл) и закрывает Playwright.


In [30]:
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd

from functions.constants import NOTEBOOK_RUN_DIR, RAW_DATA_DIR
from functions.io import load_tabular
from functions.scraper import ScrapeConfig, run_scrape

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
print("Logging configured.")


Logging configured.


## 2. Проверяем, что текущий датасет на месте

Старые файлы остаются, как есть — мы только читаем их, чтобы убедиться,
что ничего не потеряно.


In [31]:
EXISTING_RAW = RAW_DATA_DIR / "autoru_raw.parquet"
if EXISTING_RAW.exists():
    existing_df = load_tabular(EXISTING_RAW)
    print(f"Существующий датасет: {EXISTING_RAW}")
    print(f"   строк: {len(existing_df):>6}, колонок: {existing_df.shape[1]}")
else:
    existing_df = None
    print(f"Существующего датасета нет: {EXISTING_RAW}")


Существующий датасет: /Users/demn/autoru_market_research/data/raw/autoru_raw.parquet
   строк:   6333, колонок: 23


## 3. Конфигурация запуска

Все выходы изолированы в `data/raw/notebook_run/`, поэтому даже при случайном
запуске исходный `autoru_raw.parquet` не пострадает.

Параметры, которые имеет смысл крутить:
- `CATALOG_URL` — фильтр каталога (марка, регион, состояние и т.п.).
- `PAGES` — сколько страниц каталога обойти.
- `USE_PLAYWRIGHT` — `True`, если `requests` ловит анти-бот стаб-страницы.
- `CONDITION_FILTER` — `all` / `used` / `new`.
- `SELLER_TYPE_FILTER` — `all` / `private` / `dealer`.
- `INTERACTIVE_CONFIRM` — `True` → парсер откроет видимое окно браузера, после каждой
  карточки покажет, какие поля **не смог** извлечь, и встанет на `Enter`
  (`s` — пропустить запись, `q` — выйти). Автоматически включает Playwright.

> ⚠️ **Важно про интерактивный режим.** Playwright sync API не работает внутри Jupyter
> (конфликт с asyncio-циклом ядра — окно молча не открывается). Поэтому при
> `INTERACTIVE_CONFIRM=True` ноутбук **не запускает парсер сам**, а сохраняет конфиг
> и печатает команду — её надо выполнить в обычном терминале (`scrape_cli.py`).

In [32]:
NOTEBOOK_RUN_DIR.mkdir(parents=True, exist_ok=True)

run_tag = datetime.now().strftime("run_%Y%m%d_%H%M%S")
RUN_DIR = NOTEBOOK_RUN_DIR / run_tag
RUN_DIR.mkdir(parents=True, exist_ok=True)

INTERACTIVE_CONFIRM = True  # True → видимое окно + пауза Enter после каждой карточки

scrape_config = ScrapeConfig(
    catalog_url="https://auto.ru/moskovskaya_oblast/cars/used/?resolution_filter=is_owners_ok&seller_group=PRIVATE&km_age_from=50000", 
    pages=1,
    output_csv=RUN_DIR / "autoru_raw.csv",
    output_parquet=RUN_DIR / "autoru_raw.parquet",
    state_file=RUN_DIR / "autoru_state.json",
    checkpoint_jsonl=RUN_DIR / "autoru_checkpoint.jsonl",
    use_playwright=False,
    condition_filter="all",
    seller_type_filter="all",
    min_delay_seconds=1.2,
    max_delay_seconds=3.0,
    challenge_cooldown_seconds=60,
    interactive_confirm=INTERACTIVE_CONFIRM,
)
print("Артефакты будут писаться в:", RUN_DIR)
if INTERACTIVE_CONFIRM:
    print("Интерактивный режим: после каждой карточки ждём Enter.")

Артефакты будут писаться в: /Users/demn/autoru_market_research/data/raw/notebook_run/run_20260525_181947
Интерактивный режим: после каждой карточки ждём Enter.


## 4. Запуск скрапинга 

Чтобы реально запустить парсер — поменяй `RUN_SCRAPE = True`.
Перед этим ещё раз убедись, что `RUN_DIR` указывает в `data/raw/notebook_run/...`,
а не в общий `data/raw/`.

**Два режима запуска:**

- `INTERACTIVE_CONFIRM = False` — обычный фон. Парсер крутится прямо в ноутбуке
  (через `requests` или headless Playwright). Лог идёт в вывод ячейки.
- `INTERACTIVE_CONFIRM = True` — видимое окно браузера + пауза `Enter` после каждой
  карточки. Запускать **из терминала**: ноутбук сохранит JSON-конфиг и напечатает
  команду `python scrape_cli.py …`. Скопируй и выполни в терминале в корне проекта.

Почему так: Playwright sync API не уживается с asyncio-циклом Jupyter — окно из
ноутбука либо не открывается, либо процесс молча падает. Запуск в отдельном
процессе через CLI решает проблему.

In [33]:
import json

RUN_SCRAPE = True  # переключи в True, если правда хочешь запустить парсер

if RUN_SCRAPE:
    assert str(RUN_DIR).startswith(str(NOTEBOOK_RUN_DIR)), (
        "RUN_DIR должен быть внутри data/raw/notebook_run/, чтобы не затереть исходный датасет"
    )

    if INTERACTIVE_CONFIRM:
        # Playwright sync API падает в Jupyter — запускаем CLI в терминале
        config_path = RUN_DIR / "scrape_config.json"
        payload = {
            "catalog_url": scrape_config.catalog_url,
            "pages": scrape_config.pages,
            "output_csv": str(scrape_config.output_csv),
            "output_parquet": str(scrape_config.output_parquet),
            "state_file": str(scrape_config.state_file),
            "checkpoint_jsonl": str(scrape_config.checkpoint_jsonl),
            "use_playwright": True,
            "condition_filter": scrape_config.condition_filter,
            "seller_type_filter": scrape_config.seller_type_filter,
            "min_delay_seconds": scrape_config.min_delay_seconds,
            "max_delay_seconds": scrape_config.max_delay_seconds,
            "challenge_cooldown_seconds": scrape_config.challenge_cooldown_seconds,
            "interactive_confirm": True,
        }
        config_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
        scraped_df = None
        print("Конфиг сохранён в:")
        print(f"  {config_path}\n")
        print("Открой терминал в корне проекта и выполни:")
        print(f"  python scrape_cli.py {config_path}")
        print("\nВ терминале появится окно Chromium и подсказка 'Enter / s / q' после каждой карточки.")
    else:
        scraped_df = run_scrape(scrape_config)
        print(f"Спарсено строк: {len(scraped_df)}")
else:
    scraped_df = None
    print("Скрапинг пропущен. Установи RUN_SCRAPE=True, чтобы запустить.")

Конфиг сохранён в:
  /Users/demn/autoru_market_research/data/raw/notebook_run/run_20260525_181947/scrape_config.json

Открой терминал в корне проекта и выполни:
  python scrape_cli.py /Users/demn/autoru_market_research/data/raw/notebook_run/run_20260525_181947/scrape_config.json

В терминале появится окно Chromium и подсказка 'Enter / s / q' после каждой карточки.


In [38]:
# 4b. Генерация батч-конфигов (для запуска в нескольких терминалах)
import sys

# Сколько батчей делим
BATCH_COUNT = 5
# Сколько страниц в каждом батче
PAGES_PER_BATCH = 100

# Вставь URL для каждого батча (кол-во должно совпадать с BATCH_COUNT)
BATCH_URLS = [
    "https://auto.ru/cars/used/?exclude_catalog_filter=mark%3DBENTLEY&exclude_catalog_filter=mark%3DDAEWOO&exclude_catalog_filter=mark%3DDATSUN&exclude_catalog_filter=mark%3DCHEVROLET&exclude_catalog_filter=mark%3DCITROEN&exclude_catalog_filter=mark%3D212&exclude_catalog_filter=mark%3DBYD&exclude_catalog_filter=mark%3DCHANGAN&exclude_catalog_filter=mark%3DEVOLUTE&exclude_catalog_filter=mark%3DFORD&exclude_catalog_filter=mark%3DBUICK&exclude_catalog_filter=mark%3DCADILLAC&exclude_catalog_filter=mark%3DAION&exclude_catalog_filter=mark%3DCHERY&exclude_catalog_filter=mark%3DCHRYSLER&exclude_catalog_filter=mark%3DDONGFENG&exclude_catalog_filter=mark%3DBELGEE&exclude_catalog_filter=mark%3DBRILLIANCE&exclude_catalog_filter=mark%3DALPINA&exclude_catalog_filter=mark%3DFOTON&exclude_catalog_filter=mark%3DCUPRA&exclude_catalog_filter=mark%3DALFA_ROMEO&exclude_catalog_filter=mark%3DFAW&exclude_catalog_filter=mark%3DASTON_MARTIN&exclude_catalog_filter=mark%3DDADI&exclude_catalog_filter=mark%3DDS&exclude_catalog_filter=mark%3DGAZ&exclude_catalog_filter=mark%3DDODGE&exclude_catalog_filter=mark%3DBMW&exclude_catalog_filter=mark%3DACURA&exclude_catalog_filter=mark%3DAITO&exclude_catalog_filter=mark%3DDAIHATSU&exclude_catalog_filter=mark%3DDERWAYS&exclude_catalog_filter=mark%3DFACEL_VEGA&exclude_catalog_filter=mark%3DBESTUNE&exclude_catalog_filter=mark%3DDW_HOWER&exclude_catalog_filter=mark%3DGAC&exclude_catalog_filter=mark%3DAUDI&exclude_catalog_filter=mark%3DBAIC&exclude_catalog_filter=mark%3DDACIA&exclude_catalog_filter=mark%3DFIAT&exclude_catalog_filter=mark%3DCHERYEXEED&exclude_catalog_filter=mark%3DABARTH&exclude_catalog_filter=mark%3DAUTOBIANCHI&exclude_catalog_filter=mark%3DBAOJUN&km_age_from=50000&resolution_filter=is_owners_ok&seller_group=PRIVATE",
    "https://auto.ru/cars/used/?exclude_catalog_filter=mark%3DBENTLEY&exclude_catalog_filter=mark%3DDAEWOO&exclude_catalog_filter=mark%3DDATSUN&exclude_catalog_filter=mark%3DCHEVROLET&exclude_catalog_filter=mark%3DCITROEN&exclude_catalog_filter=mark%3D212&exclude_catalog_filter=mark%3DBYD&exclude_catalog_filter=mark%3DCHANGAN&exclude_catalog_filter=mark%3DEVOLUTE&exclude_catalog_filter=mark%3DFORD&exclude_catalog_filter=mark%3DBUICK&exclude_catalog_filter=mark%3DCADILLAC&exclude_catalog_filter=mark%3DAION&exclude_catalog_filter=mark%3DCHERY&exclude_catalog_filter=mark%3DCHRYSLER&exclude_catalog_filter=mark%3DDONGFENG&exclude_catalog_filter=mark%3DBELGEE&exclude_catalog_filter=mark%3DBRILLIANCE&exclude_catalog_filter=mark%3DALPINA&exclude_catalog_filter=mark%3DFOTON&exclude_catalog_filter=mark%3DCUPRA&exclude_catalog_filter=mark%3DALFA_ROMEO&exclude_catalog_filter=mark%3DFAW&exclude_catalog_filter=mark%3DASTON_MARTIN&exclude_catalog_filter=mark%3DDADI&exclude_catalog_filter=mark%3DDS&exclude_catalog_filter=mark%3DGAZ&exclude_catalog_filter=mark%3DDODGE&exclude_catalog_filter=mark%3DBMW&exclude_catalog_filter=mark%3DACURA&exclude_catalog_filter=mark%3DAITO&exclude_catalog_filter=mark%3DDAIHATSU&exclude_catalog_filter=mark%3DDERWAYS&exclude_catalog_filter=mark%3DFACEL_VEGA&exclude_catalog_filter=mark%3DBESTUNE&exclude_catalog_filter=mark%3DDW_HOWER&exclude_catalog_filter=mark%3DGAC&exclude_catalog_filter=mark%3DAUDI&exclude_catalog_filter=mark%3DBAIC&exclude_catalog_filter=mark%3DDACIA&exclude_catalog_filter=mark%3DFIAT&exclude_catalog_filter=mark%3DCHERYEXEED&exclude_catalog_filter=mark%3DABARTH&exclude_catalog_filter=mark%3DAUTOBIANCHI&exclude_catalog_filter=mark%3DBAOJUN&km_age_from=50000",
    "https://auto.ru/cars/used/?exclude_catalog_filter=mark%3DBENTLEY&exclude_catalog_filter=mark%3DDAEWOO&exclude_catalog_filter=mark%3DDATSUN&exclude_catalog_filter=mark%3DCHEVROLET&exclude_catalog_filter=mark%3DCITROEN&exclude_catalog_filter=mark%3D212&exclude_catalog_filter=mark%3DBYD&exclude_catalog_filter=mark%3DCHANGAN&exclude_catalog_filter=mark%3DEVOLUTE&exclude_catalog_filter=mark%3DFORD&exclude_catalog_filter=mark%3DBUICK&exclude_catalog_filter=mark%3DCADILLAC&exclude_catalog_filter=mark%3DAION&exclude_catalog_filter=mark%3DCHERY&exclude_catalog_filter=mark%3DCHRYSLER&exclude_catalog_filter=mark%3DDONGFENG&exclude_catalog_filter=mark%3DBELGEE&exclude_catalog_filter=mark%3DBRILLIANCE&exclude_catalog_filter=mark%3DALPINA&exclude_catalog_filter=mark%3DFOTON&exclude_catalog_filter=mark%3DCUPRA&exclude_catalog_filter=mark%3DALFA_ROMEO&exclude_catalog_filter=mark%3DFAW&exclude_catalog_filter=mark%3DASTON_MARTIN&exclude_catalog_filter=mark%3DDADI&exclude_catalog_filter=mark%3DDS&exclude_catalog_filter=mark%3DGAZ&exclude_catalog_filter=mark%3DDODGE&exclude_catalog_filter=mark%3DBMW&exclude_catalog_filter=mark%3DACURA&exclude_catalog_filter=mark%3DAITO&exclude_catalog_filter=mark%3DDAIHATSU&exclude_catalog_filter=mark%3DDERWAYS&exclude_catalog_filter=mark%3DFACEL_VEGA&exclude_catalog_filter=mark%3DBESTUNE&exclude_catalog_filter=mark%3DDW_HOWER&exclude_catalog_filter=mark%3DGAC&exclude_catalog_filter=mark%3DAUDI&exclude_catalog_filter=mark%3DBAIC&exclude_catalog_filter=mark%3DDACIA&exclude_catalog_filter=mark%3DFIAT&exclude_catalog_filter=mark%3DCHERYEXEED&exclude_catalog_filter=mark%3DABARTH&exclude_catalog_filter=mark%3DAUTOBIANCHI&exclude_catalog_filter=mark%3DBAOJUN&km_age_from=50000&seller_group=PRIVATE&sort=year-asc",
    "https://auto.ru/cars/used/?exclude_catalog_filter=mark%3DBENTLEY&exclude_catalog_filter=mark%3DDAEWOO&exclude_catalog_filter=mark%3DDATSUN&exclude_catalog_filter=mark%3DCHEVROLET&exclude_catalog_filter=mark%3DCITROEN&exclude_catalog_filter=mark%3D212&exclude_catalog_filter=mark%3DBYD&exclude_catalog_filter=mark%3DCHANGAN&exclude_catalog_filter=mark%3DEVOLUTE&exclude_catalog_filter=mark%3DFORD&exclude_catalog_filter=mark%3DBUICK&exclude_catalog_filter=mark%3DCADILLAC&exclude_catalog_filter=mark%3DAION&exclude_catalog_filter=mark%3DCHERY&exclude_catalog_filter=mark%3DCHRYSLER&exclude_catalog_filter=mark%3DDONGFENG&exclude_catalog_filter=mark%3DBELGEE&exclude_catalog_filter=mark%3DBRILLIANCE&exclude_catalog_filter=mark%3DALPINA&exclude_catalog_filter=mark%3DFOTON&exclude_catalog_filter=mark%3DCUPRA&exclude_catalog_filter=mark%3DALFA_ROMEO&exclude_catalog_filter=mark%3DFAW&exclude_catalog_filter=mark%3DASTON_MARTIN&exclude_catalog_filter=mark%3DDADI&exclude_catalog_filter=mark%3DDS&exclude_catalog_filter=mark%3DGAZ&exclude_catalog_filter=mark%3DDODGE&exclude_catalog_filter=mark%3DBMW&exclude_catalog_filter=mark%3DACURA&exclude_catalog_filter=mark%3DAITO&exclude_catalog_filter=mark%3DDAIHATSU&exclude_catalog_filter=mark%3DDERWAYS&exclude_catalog_filter=mark%3DFACEL_VEGA&exclude_catalog_filter=mark%3DBESTUNE&exclude_catalog_filter=mark%3DDW_HOWER&exclude_catalog_filter=mark%3DGAC&exclude_catalog_filter=mark%3DAUDI&exclude_catalog_filter=mark%3DBAIC&exclude_catalog_filter=mark%3DDACIA&exclude_catalog_filter=mark%3DFIAT&exclude_catalog_filter=mark%3DCHERYEXEED&exclude_catalog_filter=mark%3DABARTH&exclude_catalog_filter=mark%3DAUTOBIANCHI&exclude_catalog_filter=mark%3DBAOJUN&km_age_from=50000&seller_group=PRIVATE&sort=year-desc",
    "https://auto.ru/cars/used/?exclude_catalog_filter=mark%3DBENTLEY&exclude_catalog_filter=mark%3DDAEWOO&exclude_catalog_filter=mark%3DDATSUN&exclude_catalog_filter=mark%3DCHEVROLET&exclude_catalog_filter=mark%3DCITROEN&exclude_catalog_filter=mark%3D212&exclude_catalog_filter=mark%3DBYD&exclude_catalog_filter=mark%3DCHANGAN&exclude_catalog_filter=mark%3DEVOLUTE&exclude_catalog_filter=mark%3DFORD&exclude_catalog_filter=mark%3DBUICK&exclude_catalog_filter=mark%3DCADILLAC&exclude_catalog_filter=mark%3DAION&exclude_catalog_filter=mark%3DCHERY&exclude_catalog_filter=mark%3DCHRYSLER&exclude_catalog_filter=mark%3DDONGFENG&exclude_catalog_filter=mark%3DBELGEE&exclude_catalog_filter=mark%3DBRILLIANCE&exclude_catalog_filter=mark%3DALPINA&exclude_catalog_filter=mark%3DFOTON&exclude_catalog_filter=mark%3DCUPRA&exclude_catalog_filter=mark%3DALFA_ROMEO&exclude_catalog_filter=mark%3DFAW&exclude_catalog_filter=mark%3DASTON_MARTIN&exclude_catalog_filter=mark%3DDADI&exclude_catalog_filter=mark%3DDS&exclude_catalog_filter=mark%3DGAZ&exclude_catalog_filter=mark%3DDODGE&exclude_catalog_filter=mark%3DBMW&exclude_catalog_filter=mark%3DACURA&exclude_catalog_filter=mark%3DAITO&exclude_catalog_filter=mark%3DDAIHATSU&exclude_catalog_filter=mark%3DDERWAYS&exclude_catalog_filter=mark%3DFACEL_VEGA&exclude_catalog_filter=mark%3DBESTUNE&exclude_catalog_filter=mark%3DDW_HOWER&exclude_catalog_filter=mark%3DGAC&exclude_catalog_filter=mark%3DAUDI&exclude_catalog_filter=mark%3DBAIC&exclude_catalog_filter=mark%3DDACIA&exclude_catalog_filter=mark%3DFIAT&exclude_catalog_filter=mark%3DCHERYEXEED&exclude_catalog_filter=mark%3DABARTH&exclude_catalog_filter=mark%3DAUTOBIANCHI&exclude_catalog_filter=mark%3DBAOJUN&km_age_from=50000&seller_group=PRIVATE&sort=proven_owner-desc",
]

if len(BATCH_URLS) != BATCH_COUNT:
    raise ValueError(f"BATCH_COUNT={BATCH_COUNT}, но URL задано {len(BATCH_URLS)}")

batch_root = RUN_DIR / "batches"
batch_root.mkdir(parents=True, exist_ok=True)

batch_configs = []
for i, url in enumerate(BATCH_URLS, start=1):
    batch_name = f"batch_{i:02d}"
    batch_dir = batch_root / batch_name
    batch_dir.mkdir(parents=True, exist_ok=True)

    config_path = batch_dir / "scrape_config.json"
    payload = {
        "catalog_url": url,
        "pages": PAGES_PER_BATCH,
        "output_csv": str(batch_dir / "autoru_raw.csv"),
        "output_parquet": str(batch_dir / "autoru_raw.parquet"),
        "state_file": str(batch_dir / "autoru_state.json"),
        "checkpoint_jsonl": str(batch_dir / "autoru_checkpoint.jsonl"),
        "use_playwright": True,
        "condition_filter": scrape_config.condition_filter,
        "seller_type_filter": scrape_config.seller_type_filter,
        "min_delay_seconds": scrape_config.min_delay_seconds,
        "max_delay_seconds": scrape_config.max_delay_seconds,
        "challenge_cooldown_seconds": scrape_config.challenge_cooldown_seconds,
        "interactive_confirm": True,
    }
    config_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    batch_configs.append((i, url, config_path))

print("Батч-конфиги созданы:")
for i, url, cfg_path in batch_configs:
    print(f"  [{i}] {cfg_path}")
    print(f"      URL: {url}")

print("\nКоманды запуска (каждую в отдельный терминал):")
for i, _, cfg_path in batch_configs:
    print(f"{sys.executable} scrape_cli.py {cfg_path}")

print("\nПримечание: 5 параллельно может увеличить число капч/403. Стабильнее 1-2 параллельно.")


Батч-конфиги созданы:
  [1] /Users/demn/autoru_market_research/data/raw/notebook_run/run_20260525_181947/batches/batch_01/scrape_config.json
      URL: https://auto.ru/cars/used/?exclude_catalog_filter=mark%3DBENTLEY&exclude_catalog_filter=mark%3DDAEWOO&exclude_catalog_filter=mark%3DDATSUN&exclude_catalog_filter=mark%3DCHEVROLET&exclude_catalog_filter=mark%3DCITROEN&exclude_catalog_filter=mark%3D212&exclude_catalog_filter=mark%3DBYD&exclude_catalog_filter=mark%3DCHANGAN&exclude_catalog_filter=mark%3DEVOLUTE&exclude_catalog_filter=mark%3DFORD&exclude_catalog_filter=mark%3DBUICK&exclude_catalog_filter=mark%3DCADILLAC&exclude_catalog_filter=mark%3DAION&exclude_catalog_filter=mark%3DCHERY&exclude_catalog_filter=mark%3DCHRYSLER&exclude_catalog_filter=mark%3DDONGFENG&exclude_catalog_filter=mark%3DBELGEE&exclude_catalog_filter=mark%3DBRILLIANCE&exclude_catalog_filter=mark%3DALPINA&exclude_catalog_filter=mark%3DFOTON&exclude_catalog_filter=mark%3DCUPRA&exclude_catalog_filter=mark%3DALFA_ROMEO&

## 5. Что в итоге

Если запуска не было, в этой ячейке мы просто покажем превью **существующего** датасета,
который используют последующие ноутбуки. Если ты только что что-то спарсил — посмотрим
свежий результат.


In [35]:
preview_df = scraped_df if scraped_df is not None else existing_df

if preview_df is None:
    print("Нет ни старого, ни нового датасета — нечего показать.")
else:
    print(f"Источник: {'свежий парсинг' if scraped_df is not None else EXISTING_RAW}")
    print(f"Строк: {len(preview_df)}, колонок: {preview_df.shape[1]}\n")
    preview_df.head(5)


Источник: /Users/demn/autoru_market_research/data/raw/autoru_raw.parquet
Строк: 6333, колонок: 23



## Что дальше

- `02_eda.ipynb` — анализ датасета (нормальность, однородность, корреляции).
- `03_preprocessing.ipynb` — пошаговая очистка и формирование признаков.
- `04_regression.ipynb` — обучение и сравнение регрессионных моделей.
